# Faruq-v3 AF2 vs AF2+FFAB2 — Kaggle Stage-1 Decision
Decision-only notebook. Attach the three completed Stage-1 Kaggle outputs for seeds 42, 123, and 2026 as Kaggle inputs.

This notebook does not train and does not access locked test. It only aggregates the six frozen `val` result JSON files and decides whether DCT efficiency testing is authorized.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
SEEDS=(42,123,2026)
def one(pattern):
    hits=sorted(INPUT.rglob(pattern))
    if len(hits)!=1: raise FileNotFoundError(f'Harus ada tepat satu {pattern}; ditemukan {hits}')
    return hits[0]
AF2=[one(f'AF2FS_seed{s}_result.json') for s in SEEDS]
FFAB2=[one(f'AF2FFAB2FS_seed{s}_result.json') for s in SEEDS]
print('INPUT RESULT PREFLIGHT PASS')
for p in AF2+FFAB2: print(p)


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time
BRANCH='codex/af2-ffab2-from-start-dct'
os.chdir(WORK); REPO=WORK/'coffee-bean-detection'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('COMMIT:',COMMIT)


In [ ]:
# Validate every input before decision.
for seed,a,b in zip(SEEDS,AF2,FFAB2):
    pa=json.loads(a.read_text(encoding='utf-8')); pb=json.loads(b.read_text(encoding='utf-8'))
    assert pa['format']=='coffee_detector.af2_ffa.from_start_arm_result.v1' and pb['format']==pa['format']
    assert pa['arm']=='AF2FS' and pb['arm']=='AF2FFAB2FS'
    assert pa['seed']==seed and pb['seed']==seed
    assert pa['initial_d0_checkpoint_sha256']==pb['initial_d0_checkpoint_sha256']
    assert pa['evaluation_split']=='val' and pb['evaluation_split']=='val'
    assert pa['test_images_accessed'] is False and pb['test_images_accessed'] is False
print('THREE-SEED CONTRACT PASS')


In [ ]:
SUMMARY=WORK/'af2_ffab2_from_start_decision.json'
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_from_start_decision',
     '--af2',*[str(p) for p in AF2],'--ffab2',*[str(p) for p in FFAB2],'--output',str(SUMMARY)]
subprocess.run(cmd,cwd=REPO,check=True)
decision=json.loads(SUMMARY.read_text(encoding='utf-8'))
assert decision['test_opened'] is False
print('\nDECISION:',decision['decision'])
print('NEXT:',decision['next'])
print('CRITERIA:',json.dumps(decision['criteria'],indent=2))
print('AGGREGATE:',json.dumps(decision['aggregate'],indent=2))
meta={'branch':BRANCH,'commit':COMMIT,'evaluation_split':'val','test_images_accessed':False,'decision':decision['decision'],'next':decision['next']}
(WORK/'af2_ffab2_from_start_decision_meta.json').write_text(json.dumps(meta,indent=2)+'\n',encoding='utf-8')
print('\nOUTPUT:',SUMMARY)
if decision['decision']=='PASS': print('DCT efficiency stage is authorized. Add this decision JSON as an input to the DCT Kaggle notebook.')
else: print('STOP: FFAB2 from-start upgrade claim rejected. Do not run DCT stage.')
print('Locked test remains closed.')
